<a href="https://colab.research.google.com/github/lukeje/robot-example/blob/main/robotcar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pybullet
!git clone https://github.com/lukeje/robot-example.git
!pip install ./robot-example

Cloning into 'robot-example'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 21 (delta 4), reused 14 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 9.00 KiB | 9.00 MiB/s, done.
Resolving deltas: 100% (4/4), done.
Processing ./robot-example
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for robot_example: filename=robot_example-2026.0-py3-none-any.whl size=2939 sha256=7f0ec6ac60335df0ef7e9d9e9ddb41f301dd765c09d549aaf0546c9128ec7d55
  Stored in directory: /root/.cache/pip/wheels/8b/27/fc/2615ba37b933fc2bcc6e0c9f9d0f1454ae4f5f41672b35a419
Successfully built robot_example


In [3]:
# adapted from https://github.com/akinami3/PybulletRobotics

import math
import time

import pybullet as pb
import pybullet_data
from robot_example import controlcar

# Connect to the display
pb.connect(pb.DIRECT)


0

In [4]:

# Set simulation parameters
pb.resetSimulation() # Reset the simulation space
pb.setAdditionalSearchPath(pybullet_data.getDataPath()) # Add path to necessary data for pybullet
pb.setGravity(0.0, 0.0, -9.8) # Set gravity as on Earth
timestep = 1.0/240.0
pb.setTimeStep(timestep) # Set the elapsed time per step
nsteps = 500 # How many steps taken for each motion

# Load the floor
plane_id = pb.loadURDF("plane.urdf")

# Set the camera position and other parameters in GUI mode
camera_distance = 2.0
camera_yaw = 0.0 # deg
camera_pitch = -20 # deg
camera_target_position = [0.0, 0.0, 0.0]
pb.resetDebugVisualizerCamera(camera_distance, camera_yaw, camera_pitch, camera_target_position)

# Load the robot
car_start_pos = [0, 0, 0.1] # Set the initial position (x, y, z)
car_start_orientation = pb.getQuaternionFromEuler([0, 0, 0]) # Set the initial orientation (roll, pitch, yaw)
car_id = pb.loadURDF("robot-example/data/PybulletRobotics/urdf/simple_two_wheel_car.urdf", car_start_pos, car_start_orientation)

# Indices for joints
RIGHT_WHEEL_JOINT_IDX = 0
LEFT_WHEEL_JOINT_IDX  = 1

# Instantiate class to define the wheel velocities
mv = controlcar.MovementVelocities(timestep,nsteps)

# Loop over wheel velocities for different states
for v in (mv.goforwards(1), mv.gobackwards(0.5), mv.turnleft(math.pi/2), mv.turnright(math.pi)):
    # Set the target velocities for each wheel
    pb.setJointMotorControl2(car_id, RIGHT_WHEEL_JOINT_IDX, pb.VELOCITY_CONTROL, targetVelocity=v[RIGHT_WHEEL_JOINT_IDX])
    pb.setJointMotorControl2(car_id, LEFT_WHEEL_JOINT_IDX,  pb.VELOCITY_CONTROL, targetVelocity=v[LEFT_WHEEL_JOINT_IDX])

    # Step the simulation forwards in time
    for _ in range(nsteps):
        pb.stepSimulation()
        time.sleep(timestep)


In [5]:
pb.disconnect()